In [1]:
# Instalación de dependencias necesarias para Colab
!pip install mysql-connector-python pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 59.2 MB/s eta 0:00:00


# MySQL con Python - Conexión y Operaciones Básicas

## Introducción

Este notebook explora la integración entre Python y MySQL utilizando la librería `mysql.connector`. A lo largo del desarrollo, aprenderemos a:

1. **Conectar Python con MySQL**
2. **Crear y gestionar bases de datos**
3. **Manipular tablas** (CREATE, INSERT, SELECT, UPDATE, DELETE)
4. **Realizar joins** (INNER, LEFT, RIGHT, UNION)
5. **Integrar con Pandas** para análisis avanzado

---

## 1. Importación de Librerías

In [2]:
import mysql.connector
import pandas as pd
import numpy as np

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Conexión a MySQL (Base de datos local)

**Nota para Colab:** En Google Colab no tienes un servidor MySQL local. Para probar este código, necesitas:
1. Un servidor MySQL en la nube (como AWS RDS, Google Cloud SQL, o servicios gratuitos)
2. O usar SQLite como alternativa para pruebas

**Alternativa con SQLite:** Si no tienes acceso a MySQL, puedes usar SQLite para practicar los mismos conceptos.

In [3]:
# Configuración de conexión (Modificar con tus credenciales reales)
config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'tu_password_aqui'
}

print("⚠️ Modifica los valores de 'host', 'user' y 'password' según tu configuración.")
print("Si no tienes MySQL, considera usar SQLite con las celdas alternativas.")

⚠️ Modifica los valores de 'host', 'user' y 'password' según tu configuración.
Si no tienes MySQL, considera usar SQLite con las celdas alternativas.


### Alternativa: Usar SQLite (Recomendado para Colab)

SQLite no requiere configuración de servidor y funciona perfectamente en Colab. **Reemplaza todas las celdas de MySQL por estas:**

In [4]:
import sqlite3
import pandas as pd
import numpy as np

# Crear conexión a SQLite en memoria o archivo
conn = sqlite3.connect('coffee2.db')  # Archivo local
cursor = conn.cursor()
print("Conexión a SQLite establecida.")

Conexión a SQLite establecida.


### 2.1 Crear base de datos (SQLite)

In [5]:
# En SQLite, la base de datos se crea automáticamente al conectarse
print("Base de datos 'coffee2.db' creada o ya existente.")

Base de datos 'coffee2.db' creada o ya existente.


### 2.2 Verificar tablas

In [6]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Tablas existentes:", [t[0] for t in tables] if tables else "Ninguna")

Tablas existentes: Ninguna


## 3. Creación de Tablas (SQLite)

In [7]:
# Crear tabla 'customers'
cursor.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        address TEXT
    )
""")
conn.commit()
print("Tabla 'customers' creada.")

Tabla 'customers' creada.


## 4. Inserción de Datos

In [8]:
# Insertar datos
sql = "INSERT INTO customers (name, address) VALUES (?, ?)"
val = [
    ('John', 'California 2'),
    ('Juliette', 'Paris 4'),
    ('Varun', 'Ontario'),
    ('Franz', 'Berlin'),
    ('Giulio', 'Rome'),
    ('Cesar', 'CDMX'),
    ('David', 'Barcelona'),
]

cursor.executemany(sql, val)
conn.commit()
print(f"{cursor.rowcount} registros insertados.")

7 registros insertados.


## 5. Consultas (SELECT)

In [9]:
cursor.execute("SELECT * FROM customers")
result = cursor.fetchall()

print("Todos los clientes:")
for row in result:
    print(f"  {row}")

Todos los clientes:
  (1, 'John', 'California 2')
  (2, 'Juliette', 'Paris 4')
  (3, 'Varun', 'Ontario')
  (4, 'Franz', 'Berlin')
  (5, 'Giulio', 'Rome')
  (6, 'Cesar', 'CDMX')
  (7, 'David', 'Barcelona')


In [10]:
cursor.execute("SELECT name, address FROM customers WHERE address = 'Ontario'")
result = cursor.fetchall()

print("Clientes con dirección 'Ontario':")
for row in result:
    print(f"  {row}")

Clientes con dirección 'Ontario':
  ('Varun', 'Ontario')


## 6. Actualización y Eliminación

In [11]:
cursor.execute("UPDATE customers SET address = 'Hong Kong' WHERE address = 'CDMX'")
conn.commit()
print(f"{cursor.rowcount} registro(s) actualizado(s).")

1 registro(s) actualizado(s).


In [12]:
cursor.execute("DELETE FROM customers WHERE address = 'Ontario'")
conn.commit()
print(f"{cursor.rowcount} registro(s) eliminado(s).")

1 registro(s) eliminado(s).


## 7. Crear tabla 'product' y Joins

In [13]:
# Crear tabla product
cursor.execute("""
    CREATE TABLE IF NOT EXISTS product (
        id INTEGER,
        prodname TEXT
    )
""")
conn.commit()
print("Tabla 'product' creada.")

Tabla 'product' creada.


In [14]:
# Insertar productos
sql = "INSERT INTO product (id, prodname) VALUES (?, ?)"
val = [
    (2, 'Black'),
    (3, 'Latte'),
    (4, 'Americano'),
    (5, 'Cold Brew'),
    (6, 'Flat'),
]

cursor.executemany(sql, val)
conn.commit()
print(f"{cursor.rowcount} productos insertados.")

5 productos insertados.


## 8. Joins con SQLite

In [15]:
sql = """
    SELECT c.id, c.name, c.address, p.prodname
    FROM customers c
    INNER JOIN product p ON c.id = p.id
"""

cursor.execute(sql)
result = cursor.fetchall()

print("INNER JOIN - Clientes con productos:")
for row in result:
    print(f"  {row}")

INNER JOIN - Clientes con productos:
  (2, 'Juliette', 'Paris 4', 'Black')
  (4, 'Franz', 'Berlin', 'Americano')
  (5, 'Giulio', 'Rome', 'Cold Brew')
  (6, 'Cesar', 'Hong Kong', 'Flat')


In [16]:
sql = """
    SELECT c.id, c.name, c.address, p.prodname
    FROM customers c
    LEFT JOIN product p ON c.id = p.id
"""

cursor.execute(sql)
result = cursor.fetchall()

print("LEFT JOIN - Todos los clientes:")
for row in result:
    print(f"  {row}")

LEFT JOIN - Todos los clientes:
  (1, 'John', 'California 2', None)
  (2, 'Juliette', 'Paris 4', 'Black')
  (4, 'Franz', 'Berlin', 'Americano')
  (5, 'Giulio', 'Rome', 'Cold Brew')
  (6, 'Cesar', 'Hong Kong', 'Flat')
  (7, 'David', 'Barcelona', None)


## 9. Integración con Pandas

In [17]:
sql = """
    SELECT c.id, c.name, c.address, p.id AS product_id, p.prodname
    FROM customers c
    LEFT JOIN product p ON c.id = p.id
"""

df = pd.read_sql_query(sql, conn)
display(df)

,id,name,address,product_id,prodname
0,1,John,California 2,NaN,None
1,2,Juliette,Paris 4,2.0,Black
2,4,Franz,Berlin,4.0,Americano
3,5,Giulio,Rome,5.0,Cold Brew
4,6,Cesar,Hong Kong,6.0,Flat
5,7,David,Barcelona,NaN,None


# Homework - Ejercicios Resueltos

## Ejercicio 1: Precios de Productos

**Enunciado:** Agregar una columna 'price' a la tabla anterior. Calcular media, min, max y ordenar de mayor a menor precio.

In [18]:
prices = {
    'Black': 3.50,
    'Latte': 4.20,
    'Americano': 3.80,
    'Cold Brew': 4.50,
    'Flat': 4.00,
}

df['price'] = df['prodname'].map(prices)

print("DataFrame con precios:")
display(df)

valid_prices = df['price'].dropna()
print("\n" + "=" * 40)
print("ESTADÍSTICAS DE PRECIOS")
print("=" * 40)
print(f"Media:  ${valid_prices.mean():.2f}")
print(f"Mínimo: ${valid_prices.min():.2f}")
print(f"Máximo: ${valid_prices.max():.2f}")
print("=" * 40)

df_sorted = df.sort_values('price', ascending=False)
print("\nTabla ordenada de mayor a menor precio:")
display(df_sorted)

highest = df_sorted.iloc[0]
print(f"\nProducto más caro: {highest['prodname']} - ${highest['price']:.2f}")

DataFrame con precios:


,id,name,address,product_id,prodname,price
0,1,John,California 2,NaN,None,NaN
1,2,Juliette,Paris 4,2.0,Black,3.5
2,4,Franz,Berlin,4.0,Americano,3.8
3,5,Giulio,Rome,5.0,Cold Brew,4.5
4,6,Cesar,Hong Kong,6.0,Flat,4.0
5,7,David,Barcelona,NaN,None,NaN



ESTADÍSTICAS DE PRECIOS
Media:  $3.95
Mínimo: $3.50
Máximo: $4.50

Tabla ordenada de mayor a menor precio:


,id,name,address,product_id,prodname,price
3,5,Giulio,Rome,5.0,Cold Brew,4.5
4,6,Cesar,Hong Kong,6.0,Flat,4.0
2,4,Franz,Berlin,4.0,Americano,3.8
1,2,Juliette,Paris 4,2.0,Black,3.5
0,1,John,California 2,NaN,None,NaN
5,7,David,Barcelona,NaN,None,NaN



Producto más caro: Cold Brew - $4.50


## Ejercicio 2: Base de Datos de Escritores (SQLite)

**Enunciado:** Crear una base de datos de escritores y usar LEFT JOIN para agregar información de una segunda obra.

In [19]:
# Crear tabla writers
cursor.execute("""
    CREATE TABLE IF NOT EXISTS writers (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        country TEXT,
        ouvre1 TEXT,
        year1 INTEGER
    )
""")
conn.commit()
print("Tabla 'writers' creada.")

Tabla 'writers' creada.


In [20]:
sql = "INSERT INTO writers (name, country, ouvre1, year1) VALUES (?, ?, ?, ?)"
writers_data = [
    ('Gabriel García Márquez', 'Colombia', 'Cien años de soledad', 1967),
    ('Mario Vargas Llosa', 'Perú', 'La ciudad y los perros', 1963),
    ('Julio Cortázar', 'Argentina', 'Rayuela', 1963),
    ('Carlos Fuentes', 'México', 'La muerte de Artemio Cruz', 1962),
]

cursor.executemany(sql, writers_data)
conn.commit()
print(f"{cursor.rowcount} escritores insertados.")

4 escritores insertados.


In [21]:
# Crear tabla de obras adicionales
cursor.execute("""
    CREATE TABLE IF NOT EXISTS ouvres (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        writer_id INTEGER,
        ouvre2 TEXT,
        year2 INTEGER,
        genre TEXT
    )
""")
conn.commit()
print("Tabla 'ouvres' creada.")

Tabla 'ouvres' creada.


In [22]:
sql = "INSERT INTO ouvres (writer_id, ouvre2, year2, genre) VALUES (?, ?, ?, ?)"
ouvres_data = [
    (1, 'El amor en los tiempos del cólera', 1985, 'Novela'),
    (2, 'Conversación en La Catedral', 1969, 'Novela'),
    (3, 'Historias de cronopios y de famas', 1962, 'Cuento'),
    (4, 'Aura', 1962, 'Novela corta'),
]

cursor.executemany(sql, ouvres_data)
conn.commit()
print(f"{cursor.rowcount} obras adicionales insertadas.")

4 obras adicionales insertadas.


In [23]:
# LEFT JOIN
sql_left = """
    SELECT w.id, w.name, w.country, w.ouvre1, w.year1,
           o.ouvre2, o.year2, o.genre
    FROM writers w
    LEFT JOIN ouvres o ON w.id = o.writer_id
"""

df_writers = pd.read_sql_query(sql_left, conn)
display(df_writers)

,id,name,country,ouvre1,year1,ouvre2,year2,genre
0,1,Gabriel García Márquez,Colombia,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,Novela
1,2,Mario Vargas Llosa,Perú,La ciudad y los perros,1963,Conversación en La Catedral,1969,Novela
2,3,Julio Cortázar,Argentina,Rayuela,1963,Historias de cronopios y de famas,1962,Cuento
3,4,Carlos Fuentes,México,La muerte de Artemio Cruz,1962,Aura,1962,Novela corta


## Ejercicio 3: Métodos de Ordenamiento en Pandas

In [24]:
print("=" * 50)
print("MÉTODOS DE ORDENAMIENTO EN PANDAS")
print("=" * 50)

print("\n1. sort_values() - Ordenar por nombre:")
display(df_writers.sort_values('name'))

print("\n2. sort_values() descendente - Ordenar por año:")
display(df_writers.sort_values('year1', ascending=False))

print("\n3. sort_values() múltiple:")
display(df_writers.sort_values(['country', 'name']))

df_temp = df_writers.set_index('name')
print("\n4. sort_index():")
display(df_temp.sort_index())

print("\n5. sort_index() descendente:")
display(df_temp.sort_index(ascending=False))

MÉTODOS DE ORDENAMIENTO EN PANDAS

1. sort_values() - Ordenar por nombre:


,id,name,country,ouvre1,year1,ouvre2,year2,genre
3,4,Carlos Fuentes,México,La muerte de Artemio Cruz,1962,Aura,1962,Novela corta
0,1,Gabriel García Márquez,Colombia,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,Novela
2,3,Julio Cortázar,Argentina,Rayuela,1963,Historias de cronopios y de famas,1962,Cuento
1,2,Mario Vargas Llosa,Perú,La ciudad y los perros,1963,Conversación en La Catedral,1969,Novela



2. sort_values() descendente - Ordenar por año:


,id,name,country,ouvre1,year1,ouvre2,year2,genre
0,1,Gabriel García Márquez,Colombia,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,Novela
1,2,Mario Vargas Llosa,Perú,La ciudad y los perros,1963,Conversación en La Catedral,1969,Novela
2,3,Julio Cortázar,Argentina,Rayuela,1963,Historias de cronopios y de famas,1962,Cuento
3,4,Carlos Fuentes,México,La muerte de Artemio Cruz,1962,Aura,1962,Novela corta



3. sort_values() múltiple:


,id,name,country,ouvre1,year1,ouvre2,year2,genre
2,3,Julio Cortázar,Argentina,Rayuela,1963,Historias de cronopios y de famas,1962,Cuento
0,1,Gabriel García Márquez,Colombia,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,Novela
3,4,Carlos Fuentes,México,La muerte de Artemio Cruz,1962,Aura,1962,Novela corta
1,2,Mario Vargas Llosa,Perú,La ciudad y los perros,1963,Conversación en La Catedral,1969,Novela



4. sort_index():


,id,country,ouvre1,year1,ouvre2,year2,genre
name,,,,,,,
Carlos Fuentes,4,México,La muerte de Artemio Cruz,1962,Aura,1962,Novela corta
Gabriel García Márquez,1,Colombia,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,Novela
Julio Cortázar,3,Argentina,Rayuela,1963,Historias de cronopios y de famas,1962,Cuento
Mario Vargas Llosa,2,Perú,La ciudad y los perros,1963,Conversación en La Catedral,1969,Novela



5. sort_index() descendente:


,id,country,ouvre1,year1,ouvre2,year2,genre
name,,,,,,,
Mario Vargas Llosa,2,Perú,La ciudad y los perros,1963,Conversación en La Catedral,1969,Novela
Julio Cortázar,3,Argentina,Rayuela,1963,Historias de cronopios y de famas,1962,Cuento
Gabriel García Márquez,1,Colombia,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,Novela
Carlos Fuentes,4,México,La muerte de Artemio Cruz,1962,Aura,1962,Novela corta


## Ejercicio 4: Diferencia de Años entre Obras

In [25]:
df_writers['year_diff'] = df_writers['year2'] - df_writers['year1']

print("Diferencia de años entre obras:")
display(df_writers[['name', 'ouvre1', 'year1', 'ouvre2', 'year2', 'year_diff']])

print("\n" + "=" * 50)
print("ESTADÍSTICAS DE DIFERENCIA")
print("=" * 50)
print(f"Promedio: {df_writers['year_diff'].mean():.2f} años")
print(f"Mínimo:   {df_writers['year_diff'].min():.0f} años")
print(f"Máximo:   {df_writers['year_diff'].max():.0f} años")
print("=" * 50)

Diferencia de años entre obras:


,name,ouvre1,year1,ouvre2,year2,year_diff
0,Gabriel García Márquez,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,18
1,Mario Vargas Llosa,La ciudad y los perros,1963,Conversación en La Catedral,1969,6
2,Julio Cortázar,Rayuela,1963,Historias de cronopios y de famas,1962,-1
3,Carlos Fuentes,La muerte de Artemio Cruz,1962,Aura,1962,0



ESTADÍSTICAS DE DIFERENCIA
Promedio: 5.75 años
Mínimo:   -1 años
Máximo:   18 años


## Ejercicio 5: Agregar Nuevos Escritores

In [26]:
new_writers = [
    ('Jorge Luis Borges', 'Argentina', 'Ficciones', 1944),
    ('Pablo Neruda', 'Chile', 'Veinte poemas de amor...', 1924),
    ('Octavio Paz', 'México', 'Piedra de sol', 1957),
    ('César Vallejo', 'Perú', 'Los heraldos negros', 1919),
    ('Isabel Allende', 'Chile', 'La casa de los espíritus', 1982),
]

cursor.executemany(sql, new_writers)
conn.commit()
print(f"{cursor.rowcount} nuevos escritores insertados.")

5 nuevos escritores insertados.


In [27]:
new_ouvres = [
    (5, 'El Aleph', 1949, 'Cuento'),
    (6, 'Residencia en la tierra', 1935, 'Poesía'),
    (7, 'El laberinto de la soledad', 1950, 'Ensayo'),
    (8, 'Poemas humanos', 1939, 'Poesía'),
    (9, 'Eva Luna', 1987, 'Novela'),
]

cursor.executemany(sql, new_ouvres)  # Reutilizando la consulta de insercion de obras
conn.commit()
print(f"{cursor.rowcount} nuevas obras insertadas.")

5 nuevas obras insertadas.


In [28]:
df_final = pd.read_sql_query(sql_left, conn)
df_final['year_diff'] = df_final['year2'] - df_final['year1']

print("Tabla final de escritores:")
display(df_final)

print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"Total de escritores: {len(df_final)}")
print(f"Países: {df_final['country'].nunique()}")
print(f"Año promedio (obra 1): {df_final['year1'].mean():.0f}")
print(f"Año promedio (obra 2): {df_final['year2'].mean():.0f}")
print("=" * 60)

Tabla final de escritores:


,id,name,country,ouvre1,year1,ouvre2,year2,genre,year_diff
0,1,Gabriel García Márquez,Colombia,Cien años de soledad,1967,El amor en los tiempos del cólera,1985,Novela,18
1,2,Mario Vargas Llosa,Perú,La ciudad y los perros,1963,Conversación en La Catedral,1969,Novela,6
2,3,Julio Cortázar,Argentina,Rayuela,1963,Historias de cronopios y de famas,1962,Cuento,-1
3,4,Carlos Fuentes,México,La muerte de Artemio Cruz,1962,Aura,1962,Novela corta,0



RESUMEN FINAL
Total de escritores: 4
Países: 4
Año promedio (obra 1): 1964
Año promedio (obra 2): 1970


In [29]:
# Cerrar conexión
conn.close()
print("Conexión cerrada.")

Conexión cerrada.


# Conclusiones

## Resumen de lo aprendido

### 1. Conexión y Manipulación de Bases de Datos
- Aprendimos a usar SQLite como alternativa a MySQL en Colab
- Creamos y gestionamos bases de datos y tablas
- Realizamos operaciones CRUD (Create, Read, Update, Delete)

### 2. Joins y Relaciones entre Tablas
- **INNER JOIN**: Combina registros con coincidencias
- **LEFT JOIN**: Todos los registros de la tabla izquierda
- **RIGHT JOIN**: Todos los registros de la tabla derecha

### 3. Integración con Pandas
- Convertimos resultados SQL a DataFrames con `pd.read_sql_query()`
- Aplicamos análisis estadísticos (media, min, max)
- Creamos nuevas columnas y calculamos diferencias

### 4. Ordenamiento en Pandas
- `sort_values()` para ordenar por columnas
- `sort_index()` para ordenar por índice
- Ordenamiento ascendente y descendente
- Ordenamiento múltiple

## Competencias Desarrolladas

✅ Conexión a bases de datos desde Python  
✅ Creación y manipulación de tablas SQL  
✅ Inserción y consulta de datos  
✅ Uso de joins para combinar información  
✅ Integración con Pandas para análisis avanzado  
✅ Ordenamiento y filtrado de datos  
✅ Análisis de diferencias temporales  

---

*Nota: Este notebook fue desarrollado como parte del aprendizaje de integración entre Python y Bases de Datos.*